# Qwen3.5: General-Purpose Chest X-Ray Report Generation

### Journal Club 2026 — Section 4: Multimodal LLMs in Medical Research

This notebook adapts the MedGemma report-generation workflow to
[`unsloth/Qwen3.5-4B-GGUF`](https://huggingface.co/unsloth/Qwen3.5-4B-GGUF), a
**general-purpose multimodal model**. It uses the Indiana University (Open-i) chest
X-ray dataset to generate structured research reports, then compares model labels with
the associated radiologist report.

The key question is not whether the model writes fluent text. It is whether its positive
claims are supported by the reference: we measure both **fabrication rate** (false
positive labels / predicted positive labels) and **omission rate** (missed true labels /
reference positive labels).

> **Research and teaching only.** Qwen3.5 is not a clinically validated radiology model.
> Do not use its outputs for patient care, diagnosis, or treatment decisions.

---
## 1. Install dependencies

Qwen3.5 is loaded through the Hugging Face `transformers` interface documented on its
model card. The first run downloads the model weights, so use a GPU runtime where
available.

In [ ]:
# run this line once then comment out
# !pip install --force-reinstall --no-cache-dir pillow

In [ ]:
RUN_INSTALLS = True

# Qwen3.5 uses a Gated DeltaNet + sparse-MoE architecture that older transformers
# releases do not recognize. If loading fails with "unrecognized model type
# qwen3_5", set TRANSFORMERS_FROM_GIT = True and re-run.
#
# bitsandbytes is pinned to the floor transformers enforces for 4-bit loading.
# An older copy is worse than none: transformers raises a confusing ImportError
# saying bitsandbytes is missing when it is merely too old.
#
# pillow / pandas / matplotlib are deliberately NOT upgraded: Colab ships working
# copies, and `pip install -U pillow` over the preinstalled version can leave a
# mixed-version tree that fails with "cannot import name '_Ink' from 'PIL._typing'".
TRANSFORMERS_FROM_GIT = False

if RUN_INSTALLS:
    import subprocess
    import sys

    transformers_spec = (
        "git+https://github.com/huggingface/transformers.git@main"
        if TRANSFORMERS_FROM_GIT else "transformers>=4.57.0"
    )
    packages = [
        "kaggle",
        transformers_spec,
        "accelerate",
        "bitsandbytes>=0.46.1",
        "rouge-score",
    ]
    # Note: no -q here. A silent failure in this cell surfaces much later as an
    # unrelated-looking ImportError, so we want the pip output visible.
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", *packages],
        capture_output=True, text=True,
    )
    tail = result.stdout.strip().splitlines()[-12:]
    print("\n".join(tail))
    if result.returncode != 0:
        print("\n--- pip FAILED ---")
        print(result.stderr.strip()[-2000:])
        raise RuntimeError("Dependency installation failed; see the output above.")

    print("\nDependencies installed.")
    print(">>> Runtime > Restart session now, then run the preflight cell. <<<")
    print("The restart is required: transformers caches which optional backends")
    print("were importable at the moment it was first imported, so a bitsandbytes")
    print("installed afterwards is invisible until the process restarts.")
else:
    print("Skipping installation. Set RUN_INSTALLS = True if an import fails.")

In [ ]:
# Preflight. Run AFTER restarting the runtime and BEFORE loading the model.
# Catches a broken environment in seconds instead of after a 10 GB download.
import importlib

MIN_VERSIONS = {"bitsandbytes": (0, 46, 1), "transformers": (4, 57, 0)}


def parse_version(text):
    """Best-effort (major, minor, patch) from a version string.

    Takes only the LEADING digits of each component, so local/build suffixes
    like "2.3.3+cu121" or "0.46.1.dev0" do not inflate the patch number.
    """
    parts = []
    for chunk in str(text).split(".")[:3]:
        digits = ""
        for char in chunk:
            if not char.isdigit():
                break
            digits += char
        parts.append(int(digits) if digits else 0)
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts)


status = []
for module in ["PIL", "torch", "transformers", "accelerate",
               "bitsandbytes", "pandas", "matplotlib", "rouge_score"]:
    try:
        loaded = importlib.import_module(module)
        version = getattr(loaded, "__version__", "n/a")
        state = "ok"
        floor = MIN_VERSIONS.get(module)
        if floor and version != "n/a" and parse_version(version) < floor:
            state = f"TOO OLD (need >= {'.'.join(map(str, floor))})"
        status.append((module, version, state))
    except Exception as exc:
        status.append((module, "?", f"BROKEN: {type(exc).__name__}: {exc}"))

print(f"{'package':<16} {'version':<14} status")
print("-" * 66)
for name, version, state in status:
    print(f"{name:<16} {version:<14} {state}")

problems = [n for n, _, s in status if s != "ok"]

# The check that actually governs 4-bit loading. transformers decides this at
# import time, so it can disagree with a plain `import bitsandbytes` above.
try:
    from transformers.utils import is_bitsandbytes_available
    bnb_usable = is_bitsandbytes_available()
except Exception as exc:
    bnb_usable = False
    print(f"\nCould not query transformers for bitsandbytes support: {exc}")

print(f"\ntransformers.is_bitsandbytes_available(): {bnb_usable}")

if not bnb_usable:
    print("\n4-bit loading will FAIL. Fix with:")
    print('    !pip install -U "bitsandbytes>=0.46.1"')
    print("then Runtime > Restart session and re-run this cell.")
    print("\nIf it is already installed and still False, the runtime was not")
    print("restarted after installing it.")
elif problems:
    print(f"\nOther problems: {', '.join(problems)}")
    print("If PIL is broken:")
    print("    !pip install --force-reinstall --no-cache-dir pillow")
    print("then restart the runtime.")
else:
    print("\nAll clear. Proceed to load the model.")

In [ ]:
import torch

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | {properties.total_memory / 1024**3:.1f} GB")
else:
    print("No GPU detected. The model can run on CPU, but image generation will be slow.")

---
## 2. Authenticate and download the IU chest X-ray dataset

Download `kaggle.json` from [Kaggle Settings](https://www.kaggle.com/settings) → **API** →
**Create New Token**. In Colab, upload that file only to a private session.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))


import os
import shutil

os.makedirs("/root/.kaggle", exist_ok=True)

if os.path.exists("/content/kaggle.json"):
    shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    print("Kaggle token placed at /root/.kaggle/kaggle.json")
elif os.path.exists("/content/kaggle_API.txt"):
    shutil.copy("/content/kaggle_API.txt", "/root/.kaggle/access_token")
    os.chmod("/root/.kaggle/access_token", 0o600)
    print("Kaggle access token placed at /root/.kaggle/access_token")
else:
    raise FileNotFoundError("Upload kaggle.json or kaggle_API.txt first.")



---
## Section 3. Download the Indiana University Chest X-Ray Dataset

Kaggle dataset identifier: `raddar/chest-xrays-indiana-university`

Dataset page: <https://www.kaggle.com/datasets/raddar/chest-xrays-indiana-university>

This is the Kaggle mirror of the **Open-i / IU X-ray** collection:
- **7,470** PNG chest X-ray images (frontal and lateral)
- **3,955** de-identified radiologist reports
- Two CSVs linking them by `uid`

Unlike the class-folder datasets used in Section 2, here every image carries a
**free-text report** — which is what makes report *generation* possible.

> **Expected download size:** ~13 GB.

In [ ]:
!kaggle datasets list -s chest-xrays-indiana-university | head

In [ ]:
!kaggle datasets download -d raddar/chest-xrays-indiana-university

In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/chest-xrays-indiana-university.zip")
EXTRACT_DIR = Path("/content/iu_xray")

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Zip file not found: {ZIP_PATH}")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print(f"Extracted {ZIP_PATH.name} to {EXTRACT_DIR}")
print("\nTop-level contents:")
for p in sorted(EXTRACT_DIR.iterdir()):
    kind = "dir " if p.is_dir() else "file"
    print(f"  [{kind}] {p.name}")

---
## 3. Build frontal image–report pairs

Each IU report can have frontal and lateral images. We retain one frontal image per
study, require non-empty `findings`, and resolve image files by basename so the notebook
survives small changes in the Kaggle mirror's directory structure.

In [ ]:
from pathlib import Path
import pandas as pd

EXTRACT_DIR = Path("/content/iu_xray")
DATA_ROOT = EXTRACT_DIR

def find_file(root, name):
    matches = list(root.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} under {root}")
    return matches[0]

def find_image_dir(root):
    candidates = [directory for directory in root.rglob("*")
                  if directory.is_dir() and next(directory.glob("*.png"), None) is not None]
    if not candidates:
        raise FileNotFoundError(f"No PNG directory found under {root}")
    return max(candidates, key=lambda directory: sum(1 for _ in directory.glob("*.png")))

def resolve_column(frame, *candidates):
    columns = {column.lower().strip(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in columns:
            return columns[candidate.lower()]
    raise KeyError(f"None of {candidates} found in {list(frame.columns)}")

reports = pd.read_csv(find_file(DATA_ROOT, "indiana_reports.csv"))
projections = pd.read_csv(find_file(DATA_ROOT, "indiana_projections.csv"))
IMAGE_DIR = find_image_dir(DATA_ROOT)
IMAGE_BY_NAME = {path.name: path for path in IMAGE_DIR.glob("*.png")}

R_UID = resolve_column(reports, "uid")
R_FINDINGS = resolve_column(reports, "findings")
R_IMPRESSION = resolve_column(reports, "impression")
R_INDICATION = resolve_column(reports, "indication")
P_UID = resolve_column(projections, "uid")
P_FILENAME = resolve_column(projections, "filename")
P_PROJECTION = resolve_column(projections, "projection")

pairs = projections.merge(reports, left_on=P_UID, right_on=R_UID, how="inner")
frontal = pairs[pairs[P_PROJECTION].astype(str).str.strip().str.lower() == "frontal"].copy()
frontal = frontal[frontal[R_FINDINGS].notna()]
frontal = frontal[frontal[R_FINDINGS].astype(str).str.strip().str.len() > 20]
frontal["image_path"] = frontal[P_FILENAME].map(lambda name: IMAGE_BY_NAME.get(Path(str(name)).name))
missing_images = frontal["image_path"].isna().sum()
frontal = frontal.dropna(subset=["image_path"]).copy()
frontal["image_path"] = frontal["image_path"].map(str)
frontal = frontal.drop_duplicates(subset=[P_UID]).reset_index(drop=True)

if frontal.empty:
    raise RuntimeError("No usable frontal image-report pairs were found.")
print(f"Usable frontal pairs: {len(frontal):,}; skipped missing images: {missing_images:,}")
frontal[[P_UID, P_FILENAME, R_INDICATION, R_FINDINGS, R_IMPRESSION]].head()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
demo = frontal.sample(n=1, random_state=SEED).iloc[0]
image = Image.open(demo["image_path"]).convert("RGB")

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title(f"IU study {demo[P_UID]} ({demo[P_PROJECTION]})")
plt.show()
print("INDICATION:", demo[R_INDICATION])
print("\nFINDINGS:\n", demo[R_FINDINGS])
print("\nIMPRESSION:\n", demo[R_IMPRESSION])

---
## 4. Load Qwen3.5 4B

We use [`unsloth/Qwen3.5-4B`](https://huggingface.co/unsloth/Qwen3.5-4B) — the
**safetensors** build, not the GGUF one. GGUF is llama.cpp's container format and cannot
be read by `from_pretrained`; it would require running a separate inference server. The
safetensors repo loads directly through the same `AutoModelForImageTextToText` path used
for MedGemma, which keeps the two notebooks directly comparable.

Two details specific to this model:

**Quantization dtype depends on your GPU.** We load in 4-bit to fit ~5B parameters into a
Colab GPU. The compute dtype must match the hardware: **T4 is Turing and has no bfloat16
support**, so it needs `float16`. A100/L4 (Ampere and later) should use `bfloat16`. The
cell below detects compute capability and picks correctly — hard-coding `bfloat16` is a
common cause of silent slowdowns or NaNs on a T4.

**Thinking mode is on by default.** Qwen3.5 emits `<think>...</think>` before its answer.
Left in place, that reasoning text lands in the `TECHNIQUE` field and corrupts every
metric downstream. We disable it at the chat-template level and strip any residue.

> This is a **general-purpose** vision-language model, not a radiology model. The
> structured prompt makes output evaluable; it does not make it clinically valid.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "unsloth/Qwen3.5-4B"
LOAD_IN_4BIT = True

# Fail fast and legibly if 4-bit was requested but is not actually usable,
# rather than letting transformers raise from three frames down inside the
# quantizer during from_pretrained().
if LOAD_IN_4BIT:
    try:
        from transformers.utils import is_bitsandbytes_available
        _bnb_ok = is_bitsandbytes_available()
    except Exception:
        _bnb_ok = False
    if not _bnb_ok:
        raise RuntimeError(
            "LOAD_IN_4BIT is True but transformers cannot use bitsandbytes.\n"
            "  Fix:  !pip install -U \"bitsandbytes>=0.46.1\"\n"
            "  Then: Runtime > Restart session, re-run preflight, and re-run this cell.\n"
            "  (Installing without restarting is not enough — transformers caches\n"
            "   backend availability at import time.)\n"
            "  To proceed without quantization instead, set LOAD_IN_4BIT = False.\n"
            "  That needs roughly 10 GB of VRAM, so an L4 or A100 rather than a T4."
        )

# Turing (T4, compute capability 7.5) has no bfloat16 support; Ampere+ does.
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability(0)
    COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} (sm_{major}x, {total_vram:.1f} GB) "
          f"-> compute dtype {COMPUTE_DTYPE}")
    if not LOAD_IN_4BIT and total_vram < 16:
        print(f"WARNING: {total_vram:.1f} GB is unlikely to fit ~10 GB of weights "
              f"plus activations. Consider LOAD_IN_4BIT = True.")
else:
    COMPUTE_DTYPE = torch.float32
    print("No GPU detected. This will be extremely slow.")

if LOAD_IN_4BIT and torch.cuda.is_available():
    model_kwargs = {
        "quantization_config": BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_use_double_quant=True,
        )
    }
else:
    model_kwargs = {"dtype": COMPUTE_DTYPE}

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    device_map="auto",
    **model_kwargs,
)
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f"\nLoaded {MODEL_ID} on {INPUT_DEVICE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e9:.2f} B")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

---
## 5. Generate a structured report

The prompt asks Qwen3.5 to separate visible image findings from the provided clinical
indication. We decode greedily for reproducibility and parse only the required sections.

In [ ]:
import re
import time

SYSTEM_PROMPT = (
    "You are assisting a research demonstration of chest radiograph report generation. "
    "Describe only image-supported findings. Do not invent clinical history, comparisons, "
    "or measurements. If uncertain, state uncertainty rather than asserting a finding."
)

REPORT_SCHEMA = """Return exactly these sections and no preamble:
TECHNIQUE: projection and image quality if assessable.
FINDINGS: lungs/pleura, cardiomediastinal silhouette, bones/soft tissues, then devices.
IMPRESSION: numbered clinically important conclusions, most important first.
"""


def build_messages(pil_image, indication=None):
    context = ""
    if indication is not None and str(indication).strip() and str(indication).lower() != "nan":
        context = ("\nCLINICAL INDICATION (context only; do not assume it is true): "
                   f"{str(indication).strip()}")
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": pil_image},
            {"type": "text", "text": REPORT_SCHEMA + context},
        ]},
    ]


THINK_BLOCK = re.compile(r"<think>.*?</think>", flags=re.DOTALL | re.IGNORECASE)


def strip_thinking(text):
    """Remove Qwen3.5 reasoning blocks, including an unterminated trailing one."""
    text = THINK_BLOCK.sub("", text)
    if "<think>" in text:          # truncated output can leave an unclosed tag
        text = text.split("<think>")[0]
    return text.replace("</think>", "").strip()


def _apply_template(messages):
    """Tokenize with thinking disabled, tolerating templates that lack the flag."""
    try:
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        )
    except TypeError:
        # Older template signature: fall back and rely on strip_thinking().
        return processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )


@torch.inference_mode()
def generate_report(pil_image, indication=None, max_new_tokens=1024):
    """Generate one structured report. Greedy decoding for reproducibility."""
    messages = build_messages(pil_image, indication)
    inputs = _apply_template(messages).to(INPUT_DEVICE)
    prompt_length = inputs["input_ids"].shape[-1]
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    decoded = processor.decode(output[0][prompt_length:], skip_special_tokens=True)
    return strip_thinking(decoded)


def parse_report(text):
    sections = {"TECHNIQUE": "", "FINDINGS": "", "IMPRESSION": ""}
    tokens = re.split(r"[#*\s]*(TECHNIQUE|FINDINGS|IMPRESSION)[*\s]*:[*\s]*",
                      text, flags=re.I)
    current = None
    for token in tokens:
        label = token.strip().upper()
        if label in sections:
            current = label
        elif current is not None:
            sections[current] = token.strip().strip("*").strip()
    return sections

In [ ]:
started = time.time()
generated_raw = generate_report(image, demo[R_INDICATION])
generated_sections = parse_report(generated_raw)
print(f"Generated in {time.time() - started:.1f} seconds\n")
for name, value in generated_sections.items():
    print(f"--- {name} ---")
    print(value or "(missing)")
    print()

---
## 6. Label-based evaluation: fabrication and omission

The reference is the radiologist's `findings` plus `impression`. We use a transparent,
keyword-and-negation proxy over a small label vocabulary:

- **Fabrication rate** = false positive labels / all predicted positive labels.
  A fabricated label is asserted by Qwen3.5 but absent or negated in the reference.
- **Omission rate** = false negative labels / all reference positive labels.
  An omission is present in the reference but not asserted by Qwen3.5.

Both are label-level research proxies, not clinical validation metrics. A label with no
predicted (or no reference) positives has an undefined denominator and is reported as
`NaN`, not zero.

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
FINDING_TERMS = {
    "cardiomegaly": ["cardiomegaly", "enlarged heart", "enlarged cardiac", "cardiac enlargement"],
    "pleural_effusion": ["pleural effusion", "effusion"],
    "consolidation": ["consolidation", "airspace opacity", "airspace disease"],
    "pneumothorax": ["pneumothorax"],
    "edema": ["pulmonary edema", "edema", "vascular congestion"],
    "atelectasis": ["atelectasis", "atelectatic"],
    "nodule_or_mass": ["nodule", "mass", "lesion"],
    "fracture": ["fracture"],
    "device_or_line": ["catheter", "picc", "tube", "pacemaker", "sternotomy", "wires"],
}
NEGATION_CUES = ("no ", "without", "absence of", "negative for", "free of", "not seen", "resolved")

def is_positive(text, terms, window=60):
    """Return True if any term is affirmed; a nearby cue negates only its clause."""
    normalized = " " + str(text).lower().replace("\n", " ") + " "
    for term in terms:
        for match in re.finditer(re.escape(term), normalized):
            prefix = normalized[max(0, match.start() - window):match.start()]
            clause = re.split(r"[.;:\n]", prefix)[-1]
            if not any(cue in clause for cue in NEGATION_CUES):
                return True
    return False

def reference_text(row):
    findings = str(row[R_FINDINGS]).strip() if pd.notna(row[R_FINDINGS]) else ""
    impression = str(row[R_IMPRESSION]).strip() if pd.notna(row[R_IMPRESSION]) else ""
    return f"FINDINGS: {findings}\n\nIMPRESSION: {impression}".strip()

def generated_text(sections):
    return f"FINDINGS: {sections['FINDINGS']}\n\nIMPRESSION: {sections['IMPRESSION']}".strip()

def evaluate_labels(reference, prediction):
    rows = []
    for label, terms in FINDING_TERMS.items():
        ref_positive = is_positive(reference, terms)
        pred_positive = is_positive(prediction, terms)
        rows.append({
            "label": label,
            "reference_positive": ref_positive,
            "predicted_positive": pred_positive,
            "fabricated": pred_positive and not ref_positive,
            "omitted": ref_positive and not pred_positive,
        })
    detail = pd.DataFrame(rows)
    predicted_count = int(detail["predicted_positive"].sum())
    reference_count = int(detail["reference_positive"].sum())
    fabricated_count = int(detail["fabricated"].sum())
    omitted_count = int(detail["omitted"].sum())
    return detail, {
        "predicted_positive_labels": predicted_count,
        "reference_positive_labels": reference_count,
        "fabricated_labels": fabricated_count,
        "omitted_labels": omitted_count,
        "fabrication_rate": fabricated_count / predicted_count if predicted_count else float("nan"),
        "omission_rate": omitted_count / reference_count if reference_count else float("nan"),
    }

In [ ]:
reference = reference_text(demo)
prediction = generated_text(generated_sections)
label_detail, case_metrics = evaluate_labels(reference, prediction)
rouge = scorer.score(reference, prediction)

print("Generated report:\n", prediction)
print("\nRadiologist reference:\n", reference)
print("\nCase metrics:")
print(pd.Series(case_metrics).to_string(float_format=lambda value: f"{value:.3f}"))
print(f"\nROUGE-L F1: {rouge['rougeL'].fmeasure:.3f}")
label_detail

---
## 7. Interactive batch evaluation

Start with a small `N_CASES`—a general-purpose model may be slow and the goal is to
inspect failure modes. The batch table stores raw output, parsed sections, ROUGE, and
both label-error rates for each study.

In [ ]:
N_CASES = 8
if N_CASES < 1:
    raise ValueError("N_CASES must be at least 1.")
N_CASES = min(N_CASES, len(frontal))
batch = frontal.sample(n=N_CASES, random_state=SEED).reset_index(drop=True)

case_results = []
label_results = []
for index, row in batch.iterrows():
    pil_image = Image.open(row["image_path"]).convert("RGB")
    started = time.time()
    raw = generate_report(pil_image, row[R_INDICATION])
    elapsed = time.time() - started
    sections = parse_report(raw)
    prediction = generated_text(sections)
    reference = reference_text(row)
    detail, metrics = evaluate_labels(reference, prediction)
    rouge = scorer.score(reference, prediction)

    detail["uid"] = row[P_UID]
    label_results.append(detail)
    case_results.append({
        "uid": row[P_UID],
        "image_path": row["image_path"],
        "indication": row[R_INDICATION],
        "generated_raw": raw,
        "generated": prediction,
        "reference": reference,
        "parsed_ok": bool(sections["FINDINGS"] and sections["IMPRESSION"]),
        "rougeL_f1": rouge["rougeL"].fmeasure,
        "seconds": elapsed,
        **metrics,
    })
    print(
        f"[{index + 1}/{N_CASES}] uid={row[P_UID]} "
        f"fabrication={metrics['fabrication_rate']:.1%} "
        f"omission={metrics['omission_rate']:.1%} "
        f"({elapsed:.1f}s)"
    )

results_df = pd.DataFrame(case_results)
labels_df = pd.concat(label_results, ignore_index=True)
results_df

In [ ]:
aggregate = {
    "predicted_positive_labels": int(labels_df["predicted_positive"].sum()),
    "reference_positive_labels": int(labels_df["reference_positive"].sum()),
    "fabricated_labels": int(labels_df["fabricated"].sum()),
    "omitted_labels": int(labels_df["omitted"].sum()),
}
aggregate["fabrication_rate"] = (
    aggregate["fabricated_labels"] / aggregate["predicted_positive_labels"]
    if aggregate["predicted_positive_labels"] else float("nan")
)
aggregate["omission_rate"] = (
    aggregate["omitted_labels"] / aggregate["reference_positive_labels"]
    if aggregate["reference_positive_labels"] else float("nan")
)

print("Batch-level label metrics (micro-averaged across all cases and labels):")
print(pd.Series(aggregate).to_string(float_format=lambda value: f"{value:.3f}"))
print("\nSchema compliance:", f"{results_df['parsed_ok'].sum()}/{len(results_df)}")
print("Mean ROUGE-L F1:", f"{results_df['rougeL_f1'].mean():.3f}")

per_label = (labels_df.groupby("label")
             .agg(reference_positive=("reference_positive", "sum"),
                  predicted_positive=("predicted_positive", "sum"),
                  fabricated=("fabricated", "sum"),
                  omitted=("omitted", "sum"))
             .sort_values(["fabricated", "omitted"], ascending=False))
per_label["fabrication_rate"] = per_label["fabricated"] / per_label["predicted_positive"].replace(0, float("nan"))
per_label["omission_rate"] = per_label["omitted"] / per_label["reference_positive"].replace(0, float("nan"))
per_label

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(results_df["fabrication_rate"].dropna(), bins=8, color="#c44e52", edgecolor="white")
axes[0].set(title="Per-case fabrication rate", xlabel="false labels / predicted labels", ylabel="cases")
axes[1].hist(results_df["omission_rate"].dropna(), bins=8, color="#4c72b0", edgecolor="white")
axes[1].set(title="Per-case omission rate", xlabel="missed labels / reference labels", ylabel="cases")
plt.tight_layout()
plt.show()

review_columns = ["uid", "rougeL_f1", "fabrication_rate", "omission_rate", "parsed_ok", "seconds"]
results_df.sort_values(["fabrication_rate", "omission_rate"], ascending=False)[review_columns]

---
## 8. Discussion prompts

1. Does Qwen3.5's fluent prose hide a high fabrication rate?
2. Which labels are most often omitted? Are they subtle, uncommon, or poorly represented
   by the keyword proxy?
3. Does ROUGE-L track either fabrication or omission? Inspect discordant cases.
4. What would a clinical evaluation require beyond this notebook—expert image rereads,
   validated labelers such as CheXbert/RadGraph, calibration, and prospective testing?
5. How does replacing a medical model with a general-purpose model change the risk
   profile, even when the prompt and dataset stay constant?

In [ ]:
OUTPUT_DIR = Path("/content/qwen35_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "qwen35_generated_reports.csv", index=False)
labels_df.to_csv(OUTPUT_DIR / "qwen35_label_evaluation.csv", index=False)
per_label.to_csv(OUTPUT_DIR / "qwen35_per_label_metrics.csv")
print(f"Saved report and metric tables to {OUTPUT_DIR}")